In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/model-dataset/Model_dataset.csv


In [2]:
!pip -q install --no-cache-dir --force-reinstall \
  "numpy==2.1.3" \
  "scipy==1.14.1" \
  "scikit-learn==1.6.1" \
  "pyarrow==15.0.2" \
  "datasets==2.21.0" \
  "transformers==4.47.1" \
  "accelerate==1.2.1" \
  "evaluate==0.4.3" \
  "iterative-stratification==0.1.9" \
  "peft==0.13.2"

print("✅ Install done. NOW: Runtime → Restart session")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 125.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 263.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 284.0 MB/s eta 0:00:00
ERROR: Cannot install numpy==2.1.3, pyarrow==15.0.2, scikit-learn==1.6.1 and scipy==1.14.1 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
✅ Install done. NOW: Runtime → Restart session


In [3]:
!pip -q install iterative-stratification==0.1.9

In [4]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
print("✅ iterstrat OK")


✅ iterstrat OK


In [5]:
import os, re, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import f1_score, precision_score, recall_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed
)

import pyarrow, datasets, transformers, sklearn, scipy

print("✅ Versions:")
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("pyarrow:", pyarrow.__version__)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("✅ Environment OK")


2026-01-13 04:40:38.733069: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768279238.917906      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768279238.978145      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

✅ Versions:
numpy: 1.26.4
scipy: 1.15.3
sklearn: 1.2.2
pyarrow: 19.0.1
datasets: 4.4.1
transformers: 4.53.3
torch: 2.6.0+cu124
✅ Environment OK


In [7]:
# ---------------------------
# ✅ Config
# ---------------------------
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

TEXT_COL = "Data"
LABEL_COLS = ["Love", "Joy", "Anger", "Surprise", "Sadness", "Fear", "Hate"]

DATA_PATH = "/kaggle/input/model-dataset/Model_dataset.csv"

# For speed vs quality:
MAX_LEN = 160          # 128 -> faster, 160 -> often better F1
EPOCHS  = 12           # 6-8 is enough with early stopping
LR      = 2e-5
WD      = 0.01
WARMUP  = 0.10

# ✅ Two-model ensemble (recommended)
MODELS = [
    "xlm-roberta-base",
    "microsoft/mdeberta-v3-base"
]

OUTDIR = "/kaggle/working/ens_focal_banglish"
os.makedirs(OUTDIR, exist_ok=True)


In [8]:
# ---------------------------
# ✅ Text Cleaning (huge impact for Banglish)
# ---------------------------
_url_re = re.compile(r"(http\S+|www\.\S+)")
_user_re = re.compile(r"@\w+")
_multi_space = re.compile(r"\s+")

def clean_text(s: str) -> str:
    s = str(s)
    s = s.strip()
    s = s.lower()
    s = _url_re.sub(" ", s)
    s = _user_re.sub(" ", s)
    # reduce char repetition: "cooool" -> "cool"
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)
    s = _multi_space.sub(" ", s).strip()
    return s

In [9]:
# ---------------------------
# ✅ Load Data
# ---------------------------
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

missing = [c for c in [TEXT_COL] + LABEL_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}\nFound: {df.columns.tolist()}")

df = df[[TEXT_COL] + LABEL_COLS].copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").map(clean_text)

for c in LABEL_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

df = df[df[TEXT_COL].str.strip().astype(bool)].reset_index(drop=True)

print("✅ Samples:", len(df))
print("✅ Labels:", LABEL_COLS)
print(df.head(2))

✅ Samples: 27809
✅ Labels: ['Love', 'Joy', 'Anger', 'Surprise', 'Sadness', 'Fear', 'Hate']
                                                Data  Love  Joy  Anger  \
0  atae best story amar darun lagacha wow smiling...     0    0      0   
1                 onek valo hoto jodi phone ta patam     0    0      0   

   Surprise  Sadness  Fear  Hate  
0         1        0     0     0  
1         0        1     0     0  


In [10]:
# ---------------------------
# ✅ Multilabel Stratified Split (70/15/15)
# ---------------------------
y = df[LABEL_COLS].values

msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(msss1.split(df, y))

train_df = df.iloc[train_idx].reset_index(drop=True)
temp_df  = df.iloc[temp_idx].reset_index(drop=True)

y_temp = temp_df[LABEL_COLS].values
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(msss2.split(temp_df, y_temp))

val_df  = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

print(f"✅ Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

✅ Train: 19466 | Val: 4171 | Test: 4172


In [11]:
# ---------------------------
# ✅ POS_WEIGHT (imbalance fix)
# ---------------------------
train_y = train_df[LABEL_COLS].values.astype(np.float32)
pos = train_y.sum(axis=0)
neg = (train_y.shape[0] - pos)
pos_weight = (neg / (pos + 1e-6))
pos_weight_t = torch.tensor(pos_weight, dtype=torch.float32)
print("✅ pos_weight:", dict(zip(LABEL_COLS, np.round(pos_weight, 2))))


✅ pos_weight: {'Love': 5.42, 'Joy': 5.42, 'Anger': 5.42, 'Surprise': 5.42, 'Sadness': 5.42, 'Fear': 5.42, 'Hate': 5.42}


In [12]:
# ---------------------------
# ✅ Metrics helpers
# ---------------------------
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics_default(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)
    preds = (probs >= 0.5).astype(int)

    micro_f1  = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1  = f1_score(labels, preds, average="macro", zero_division=0)
    micro_p   = precision_score(labels, preds, average="micro", zero_division=0)
    micro_r   = recall_score(labels, preds, average="micro", zero_division=0)
    subset_acc = (preds == labels).all(axis=1).mean()
    label_acc  = (preds == labels).mean()

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "micro_precision": micro_p,
        "micro_recall": micro_r,
        "subset_accuracy": subset_acc,
        "label_accuracy": label_acc,
    }

def find_best_thresholds_from_probs(probs, y_true, label_cols):
    best_thr = []
    for i in range(y_true.shape[1]):
        best_t, best_f1 = 0.5, -1
        for t in np.linspace(0.20, 0.80, 25):
            y_pred = (probs[:, i] >= t).astype(int)
            f1 = f1_score(y_true[:, i], y_pred, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = float(t)
        best_thr.append(best_t)
    best_thr = np.array(best_thr, dtype=np.float32)
    print("✅ Best thresholds:", dict(zip(label_cols, np.round(best_thr, 3))))
    return best_thr

def evaluate_probs_with_thresholds(probs, y_true, thresholds):
    y_pred = (probs >= thresholds.reshape(1, -1)).astype(int)
    return {
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "subset_accuracy": (y_pred == y_true).all(axis=1).mean(),
        "label_accuracy": (y_pred == y_true).mean(),
    }


In [13]:
# ---------------------------
# ✅ Focal Loss (works great on imbalance)
# ---------------------------
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        # keep as buffer, but we'll move it to correct device at forward
        self.register_buffer("pos_weight", pos_weight)

    def forward(self, logits, targets):
        # ✅ make sure targets same dtype/device as logits
        targets = targets.to(device=logits.device, dtype=logits.dtype)

        # ✅ move pos_weight to same device/dtype as logits
        pw = None
        if self.pos_weight is not None:
            pw = self.pos_weight.to(device=logits.device, dtype=logits.dtype)

        bce = torch.nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction="none", pos_weight=pw
        )

        p = torch.sigmoid(logits)
        pt = p * targets + (1 - p) * (1 - targets)
        focal = (1 - pt).pow(self.gamma) * bce
        return focal.mean()



In [15]:
class MultilabelFocalTrainer(Trainer):
    def __init__(self, *args, gamma=2.0, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal = FocalLoss(gamma=gamma, pos_weight=pos_weight)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").to(torch.float32)   # ✅ float labels
        outputs = model(**inputs)
        loss = self.focal(outputs.logits, labels)         # ✅ focal handles device
        return (loss, outputs) if return_outputs else loss


In [16]:

# ---------------------------
# ✅ Build HF datasets per model (tokenizer differs)
# ---------------------------
def make_hf_ds(tokenizer, train_df, val_df, test_df):
    def to_ds(pdf):
        return Dataset.from_pandas(pdf, preserve_index=False)

    train_ds = to_ds(train_df)
    val_ds   = to_ds(val_df)
    test_ds  = to_ds(test_df)

    def tokenize_fn(batch):
        return tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)

    train_ds = train_ds.map(tokenize_fn, batched=True)
    val_ds   = val_ds.map(tokenize_fn, batched=True)
    test_ds  = test_ds.map(tokenize_fn, batched=True)

    def add_labels(batch):
        labels = np.stack([batch[c] for c in LABEL_COLS], axis=1).astype(np.float32)
        batch["labels"] = labels
        return batch

    train_ds = train_ds.map(add_labels, batched=True)
    val_ds   = val_ds.map(add_labels, batched=True)
    test_ds  = test_ds.map(add_labels, batched=True)

    cols_to_keep = ["input_ids", "attention_mask", "labels"]
    train_ds.set_format(type="torch", columns=cols_to_keep)
    val_ds.set_format(type="torch", columns=cols_to_keep)
    test_ds.set_format(type="torch", columns=cols_to_keep)
    return train_ds, val_ds, test_ds


In [18]:
# ---------------------------
# ✅ Train one model and return probs (UPDATED + FIXED)
# ---------------------------
def train_one_model(model_name, idx):
    print("\n" + "=" * 70)
    print(f"🚀 Training model {idx+1}/{len(MODELS)}: {model_name}")
    print("=" * 70)

    is_deberta = "deberta" in model_name.lower()

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_ds, val_ds, test_ds = make_hf_ds(tokenizer, train_df, val_df, test_df)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    config = AutoConfig.from_pretrained(model_name)
    config.num_labels = len(LABEL_COLS)
    config.problem_type = "multi_label_classification"

    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)

    # ✅ IMPORTANT: DeBERTa + custom loss => gradient checkpointing OFF (fixes double-backward)
    model.config.use_cache = False
    if is_deberta:
        if hasattr(model, "gradient_checkpointing_disable"):
            model.gradient_checkpointing_disable()
        grad_ckpt = False
        train_bs = 8
        gas = 4   # effective batch = 32
    else:
        model.gradient_checkpointing_enable()
        grad_ckpt = True
        train_bs = 16
        gas = 2   # effective batch = 32

    out_dir = f"{OUTDIR}/{model_name.replace('/','_')}"

    args = TrainingArguments(
        output_dir=out_dir,
        seed=SEED,

        num_train_epochs=EPOCHS,
        learning_rate=LR,
        weight_decay=WD,
        warmup_ratio=WARMUP,
        lr_scheduler_type="cosine",

        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=64,
        gradient_accumulation_steps=gas,

        fp16=True,
        gradient_checkpointing=grad_ckpt,   # ✅ must match model setting

        logging_steps=200,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,

        load_best_model_at_end=True,
        metric_for_best_model="micro_f1",
        greater_is_better=True,

        report_to="none"
    )

    trainer = MultilabelFocalTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        compute_metrics=compute_metrics_default,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        gamma=2.0,
        pos_weight=pos_weight_t
    )

    trainer.train()
    print("✅ Best checkpoint:", trainer.state.best_model_checkpoint)

    val_probs = sigmoid(trainer.predict(val_ds).predictions)
    test_probs = sigmoid(trainer.predict(test_ds).predictions)

    # ✅ Save model/tokenizer
    save_dir = f"/kaggle/working/saved_{model_name.replace('/','_')}"
    os.makedirs(save_dir, exist_ok=True)
    trainer.save_model(save_dir)
    tokenizer.save_pretrained(save_dir)
    print("✅ Saved:", save_dir)

    return val_probs, test_probs


In [19]:
# ---------------------------
# ✅ Train all models + Ensemble
# ---------------------------
val_probs_list, test_probs_list = [], []

for i, m in enumerate(MODELS):
    vp, tp = train_one_model(m, i)
    val_probs_list.append(vp)
    test_probs_list.append(tp)

# Ensemble: average probs
ens_val_probs  = np.mean(np.stack(val_probs_list, axis=0), axis=0)
ens_test_probs = np.mean(np.stack(test_probs_list, axis=0), axis=0)

y_val  = val_df[LABEL_COLS].values.astype(int)
y_test = test_df[LABEL_COLS].values.astype(int)

# Threshold tuning on ensemble
best_thr = find_best_thresholds_from_probs(ens_val_probs, y_val, LABEL_COLS)

print("\n✅ VAL (Ensemble + threshold) metrics:")
val_metrics = evaluate_probs_with_thresholds(ens_val_probs, y_val, best_thr)
for k,v in val_metrics.items():
    print(f"{k}: {v:.4f}")

print("\n✅ TEST (Ensemble + threshold) metrics:")
test_metrics = evaluate_probs_with_thresholds(ens_test_probs, y_test, best_thr)
for k,v in test_metrics.items():
    print(f"{k}: {v:.4f}")

# Save thresholds + ensemble meta
final_dir = "/kaggle/working/final_ensemble"
os.makedirs(final_dir, exist_ok=True)
np.save(os.path.join(final_dir, "best_thresholds.npy"), best_thr)
with open(os.path.join(final_dir, "models_used.txt"), "w") as f:
    f.write("\n".join(MODELS))

print("\n✅ Final ensemble saved to:", final_dir)
print("✅ Files:", os.listdir(final_dir))


🚀 Training model 1/2: xlm-roberta-base


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/19466 [00:00<?, ? examples/s]

... (22 outputs left out)

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall,Subset Accuracy,Label Accuracy
1,0.246000,0.229410,0.430069,0.450502,0.298724,0.767554,0.042436,0.683461
2,0.207500,0.207423,0.508327,0.510236,0.375618,0.786044,0.132103,0.763400
3,0.177700,0.182402,0.551037,0.559113,0.413005,0.827647,0.187245,0.790150
4,0.153800,0.178038,0.580262,0.584092,0.449607,0.817962,0.267562,0.815871
5,0.134300,0.193554,0.591320,0.595580,0.469243,0.799252,0.315032,0.828099
6,0.116200,0.192887,0.604814,0.609859,0.486468,0.799252,0.357468,0.837483
7,0.099300,0.206709,0.625869,0.629450,0.512943,0.802553,0.382882,0.850704
8,0.087700,0.216867,0.637343,0.641170,0.529403,0.800572,0.402062,0.858239
9,0.079700,0.239292,0.645185,0.647658,0.545759,0.788906,0.435387,0.864986
10,0.071600,0.250737,0.647813,0.650458,0.554441,0.779001,0.452409,0.868206


✅ Best checkpoint: /kaggle/working/ens_focal_banglish/microsoft_mdeberta-v3-base/checkpoint-7308


✅ Saved: /kaggle/working/saved_microsoft_mdeberta-v3-base
✅ Best thresholds: {'Love': 0.6, 'Joy': 0.55, 'Anger': 0.525, 'Surprise': 0.675, 'Sadness': 0.6, 'Fear': 0.65, 'Hate': 0.55}

✅ VAL (Ensemble + threshold) metrics:
micro_f1: 0.6908
macro_f1: 0.6970
micro_precision: 0.6646
micro_recall: 0.7191
subset_accuracy: 0.5411
label_accuracy: 0.8998

✅ TEST (Ensemble + threshold) metrics:
micro_f1: 0.6881
macro_f1: 0.6946
micro_precision: 0.6565
micro_recall: 0.7229
subset_accuracy: 0.5403
label_accuracy: 0.8979

✅ Final ensemble saved to: /kaggle/working/final_ensemble
✅ Files: ['models_used.txt', 'best_thresholds.npy']


In [20]:
# ============================
# ✅ Overall (Full) Accuracy (without subset accuracy)
# Works for multi-label classification
# ============================

# 1) Build final test predictions using your tuned thresholds
y_pred_test = (ens_test_probs >= best_thr.reshape(1, -1)).astype(int)

# 2) "Overall" accuracy in multi-label = label-wise accuracy (micro over all labels)
#    (same as: (y_pred_test == y_test).mean())
overall_label_accuracy = (y_pred_test == y_test).mean()

# 3) Per-label accuracy + Macro accuracy (average of label accuracies)
per_label_acc = (y_pred_test == y_test).mean(axis=0)
macro_label_accuracy = per_label_acc.mean()

print("✅ Overall label-wise accuracy (micro over all labels):", round(float(overall_label_accuracy), 6))
print("✅ Macro label accuracy (mean of per-label accuracies):", round(float(macro_label_accuracy), 6))

print("\nPer-label accuracy:")
for name, acc in zip(LABEL_COLS, per_label_acc):
    print(f"  {name:10s}: {float(acc):.6f}")

# (Optional) Hamming Loss (lower is better)
hamming_loss = 1.0 - overall_label_accuracy
print("\n✅ Hamming loss:", round(float(hamming_loss), 6))


✅ Overall label-wise accuracy (micro over all labels): 0.897891
✅ Macro label accuracy (mean of per-label accuracies): 0.897891

Per-label accuracy:
  Love      : 0.910115
  Joy       : 0.863615
  Anger     : 0.841563
  Surprise  : 0.946069
  Sadness   : 0.877996
  Fear      : 0.947267
  Hate      : 0.898610

✅ Hamming loss: 0.102109
